# Governance Table Setup

**Run once** to create the `Governance` schema and all governance tables in the `DW_Fabric`
Warehouse, plus seed default configuration.

Requires: access to `DW_Fabric` (Warehouse), and a default Lakehouse attached (only for
reading the Xebia logo PNG file — Warehouses don't have a Files area).

### Tables created (all in the `Governance` schema)
| Table | Purpose |
|-------|---------|
| `cleanup_tracker` | Warning lifecycle state per item |
| `cleanup_audit_log` | Immutable log of all actions |
| `governance_config` | Configurable thresholds and rules |
| `email_outbox` | Generated emails queue for pipeline Outlook activity |
| `deleted_items_archive` | Full metadata snapshot of every item, written just before it's permanently deleted |
| `workspace_inventory_snapshot` | Raw inventory scan output (created here explicitly — Warehouse tables, unlike Delta, don't auto-create on first write) |

All string-typed columns are `VARCHAR` — this project stores everything as text (every
consumer already does `CAST(... AS INT)` etc. where needed), so this matches the Lakehouse
version's convention rather than introducing new typed columns. `NVARCHAR`/`NCHAR` are used
intentionally nowhere here: Fabric Data Warehouse does not support them at all — it stores
Unicode text in `VARCHAR` columns under a UTF-8 collation instead.

## 1. Connect to the Warehouse

In [ ]:
import pyodbc, struct, notebookutils
import pandas as pd

WAREHOUSE_SQL_ENDPOINT = "<WAREHOUSE_SQL_ENDPOINT>"
WAREHOUSE_DATABASE     = "DW_Fabric"
GOVERNANCE_SCHEMA      = "Governance"

def get_warehouse_connection():
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    conn_str = (
        f"Driver={{ODBC Driver 18 for SQL Server}};"
        f"Server={WAREHOUSE_SQL_ENDPOINT};"
        f"Database={WAREHOUSE_DATABASE};"
        f"Encrypt=Yes;TrustServerCertificate=No"
    )
    return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})

conn = get_warehouse_connection()
print("Connected to DW_Fabric.")

## 2. Create the Governance schema

In [ ]:
cursor = conn.cursor()
cursor.execute("""
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'Governance')
    EXEC('CREATE SCHEMA Governance');
""")
conn.commit()
print("\u2714 Governance schema ready")

## 3. cleanup_tracker

In [ ]:
cursor.execute("""
IF OBJECT_ID('Governance.cleanup_tracker', 'U') IS NOT NULL
    DROP TABLE Governance.cleanup_tracker;

CREATE TABLE Governance.cleanup_tracker (
    item_id                  VARCHAR(50)   NOT NULL,
    item_name                VARCHAR(MAX)  NULL,
    item_type                VARCHAR(100)  NULL,
    owner_email               VARCHAR(320)  NULL,
    cleanup_score             VARCHAR(20)   NULL,
    first_flagged_date        VARCHAR(50)   NULL,
    warning_count             VARCHAR(20)   NULL,
    warning_1_date            VARCHAR(50)   NULL,
    warning_2_date            VARCHAR(50)   NULL,
    warning_3_date            VARCHAR(50)   NULL,
    status                    VARCHAR(50)   NULL,
    deleted_date              VARCHAR(50)   NULL,
    resolved_date             VARCHAR(50)   NULL,
    exemption_reason          VARCHAR(MAX)  NULL,
    last_updated              VARCHAR(50)   NULL,
    pipeline_run_id           VARCHAR(50)   NULL,
    workspace_id              VARCHAR(50)   NULL,
    deletion_notified         VARCHAR(20)   NULL,
    manual_action_notified    VARCHAR(20)   NULL
);
""")
conn.commit()
print("\u2714 cleanup_tracker created")

## 4. cleanup_audit_log

In [ ]:
cursor.execute("""
IF OBJECT_ID('Governance.cleanup_audit_log', 'U') IS NOT NULL
    DROP TABLE Governance.cleanup_audit_log;

CREATE TABLE Governance.cleanup_audit_log (
    audit_id         VARCHAR(50)   NOT NULL,
    [timestamp]      VARCHAR(50)   NULL,
    pipeline_run_id  VARCHAR(50)   NULL,
    item_id          VARCHAR(50)   NULL,
    item_name        VARCHAR(MAX)  NULL,
    item_type        VARCHAR(100)  NULL,
    owner_email      VARCHAR(320)  NULL,
    action           VARCHAR(100)  NULL,
    detail            VARCHAR(MAX)  NULL,
    workspace_id     VARCHAR(50)   NULL
);
""")
conn.commit()
print("\u2714 cleanup_audit_log created")

## 5. email_outbox

In [ ]:
cursor.execute("""
IF OBJECT_ID('Governance.email_outbox', 'U') IS NOT NULL
    DROP TABLE Governance.email_outbox;

CREATE TABLE Governance.email_outbox (
    email_id         VARCHAR(50)   NOT NULL,
    pipeline_run_id  VARCHAR(50)   NULL,
    email_type       VARCHAR(100)  NULL,
    recipient        VARCHAR(320)  NULL,
    subject          VARCHAR(MAX)  NULL,
    body_html        VARCHAR(MAX)  NULL,
    status           VARCHAR(20)   NULL,
    created_at       VARCHAR(50)   NULL,
    workspace_id     VARCHAR(50)   NULL,
    sent_at          VARCHAR(50)   NULL
);
""")
conn.commit()
print("\u2714 email_outbox created")

## 6. deleted_items_archive

In [ ]:
# Full metadata snapshot of every item at the moment it's permanently deleted.
# Written by Governance_Auto_Delete BEFORE the DELETE API call, so a wrongly-deleted
# item's complete governance record (owner, scores, dates, flags) is preserved even
# though the item itself is gone. This is what backs the "contact the administrator
# within 48 hours" recovery window promised in the deletion-confirmation email.
cursor.execute("""
IF OBJECT_ID('Governance.deleted_items_archive', 'U') IS NOT NULL
    DROP TABLE Governance.deleted_items_archive;

CREATE TABLE Governance.deleted_items_archive (
    item_id                     VARCHAR(50)   NOT NULL,
    item_name                   VARCHAR(MAX)  NULL,
    item_type                   VARCHAR(100)  NULL,
    workspace_id                VARCHAR(50)   NULL,
    workspace_name               VARCHAR(MAX)  NULL,
    owner_email                  VARCHAR(320)  NULL,
    description                  VARCHAR(MAX)  NULL,
    web_url                      VARCHAR(MAX)  NULL,
    created_date                 VARCHAR(50)   NULL,
    last_modified                VARCHAR(50)   NULL,
    last_used_date                VARCHAR(50)   NULL,
    cleanup_candidate_score       VARCHAR(20)   NULL,
    is_stale                     VARCHAR(20)   NULL,
    is_unused_artifact            VARCHAR(20)   NULL,
    has_missing_owner             VARCHAR(20)   NULL,
    is_duplicate_name             VARCHAR(20)   NULL,
    is_orphaned_model             VARCHAR(20)   NULL,
    is_orphaned_endpoint          VARCHAR(20)   NULL,
    warning_1_date                VARCHAR(50)   NULL,
    warning_2_date                VARCHAR(50)   NULL,
    warning_3_date                VARCHAR(50)   NULL,
    deleted_by_pipeline_run_id    VARCHAR(50)   NULL,
    archived_at                   VARCHAR(50)   NULL
);
""")
conn.commit()
print("\u2714 deleted_items_archive created")

## 7. workspace_inventory_snapshot

In [ ]:
# The 27-column raw inventory table. Under Delta, this table was created implicitly on the
# Inventory notebook's first write (mode="append" auto-creates a Delta table). Warehouse
# tables don't auto-create on INSERT, so it's created explicitly here instead.
cursor.execute("""
IF OBJECT_ID('Governance.workspace_inventory_snapshot', 'U') IS NOT NULL
    DROP TABLE Governance.workspace_inventory_snapshot;

CREATE TABLE Governance.workspace_inventory_snapshot (
    id                        VARCHAR(50)   NULL,
    name                      VARCHAR(MAX)  NULL,
    [type]                    VARCHAR(100)  NULL,
    description                VARCHAR(MAX)  NULL,
    web_url                    VARCHAR(MAX)  NULL,
    workspace_id               VARCHAR(50)   NULL,
    workspace_name              VARCHAR(MAX)  NULL,
    capacity_id                 VARCHAR(50)   NULL,
    created_by                  VARCHAR(320)  NULL,
    modified_by                 VARCHAR(320)  NULL,
    created_date                 VARCHAR(50)   NULL,
    last_modified                VARCHAR(50)   NULL,
    last_used_date                VARCHAR(50)   NULL,
    access_count_30d              VARCHAR(20)   NULL,
    unique_users_30d              VARCHAR(20)   NULL,
    is_unused_artifact             VARCHAR(20)   NULL,
    days_since_modified             VARCHAR(20)   NULL,
    days_since_last_used            VARCHAR(20)   NULL,
    state                          VARCHAR(50)   NULL,
    is_stale                       VARCHAR(20)   NULL,
    has_missing_owner               VARCHAR(20)   NULL,
    is_duplicate_name               VARCHAR(20)   NULL,
    is_orphaned_model                VARCHAR(20)   NULL,
    is_orphaned_endpoint             VARCHAR(20)   NULL,
    cleanup_candidate_score           VARCHAR(20)   NULL,
    snapshot_id                      VARCHAR(50)   NULL,
    snapshot_time_utc                 VARCHAR(50)   NULL
);
""")
conn.commit()
print("\u2714 workspace_inventory_snapshot created")

## 8. governance_config

In [ ]:
# NOTE: admin_email has a hardcoded fallback in Email_Generator/Auto_Delete/Failure_Notifier
# if this key is ever missing — those notebooks log a loud CRITICAL warning when that
# fallback kicks in rather than failing silently.
cursor.execute("""
IF OBJECT_ID('Governance.governance_config', 'U') IS NOT NULL
    DROP TABLE Governance.governance_config;

CREATE TABLE Governance.governance_config (
    config_key    VARCHAR(200)  NOT NULL,
    config_value  VARCHAR(MAX)  NULL,
    description   VARCHAR(MAX)  NULL
);
""")
conn.commit()

config_defaults = [
    ("cleanup_score_threshold", "30",    "Min score to enter cleanup workflow"),
    ("stale_cutoff_days",       "90",    "Days to flag as stale"),
    ("warnings_before_delete",  "3",     "Warnings before auto-delete"),
    ("days_between_warnings",   "1",     "Min days between warnings (1=testing, 7=production)"),
    ("admin_email",             "<ADMIN_EMAIL>", "Dashboard report recipient"),
    ("pipeline_schedule",       "daily_9am", "When pipeline runs"),
    ("protected_types",         "Lakehouse,Warehouse,Environment,SQLEndpoint", "Types never auto-deleted"),
    ("protected_items",         "",       "Comma-separated item IDs never deleted"),
    ("enable_auto_delete",      "false",  "Master switch for deletion (start disabled)"),
    ("workspace_ids",           "<WORKSPACE_ID>", "Comma-separated workspace IDs"),
    ("logo_url",                "",       "Xebia logo URL or base64 for emails"),
    ("test_mode_recipient",     "",       "Redirect ALL emails here for testing (blank to disable)"),
    ("dry_run",                 "true",   "Log actions without executing (true for testing)"),
    ("pipeline_failure_notify_email", "", "Recipient for pipeline activity-failure alerts (blank = falls back to admin_email)"),
]

cursor.setinputsizes([(pyodbc.SQL_VARCHAR, 0, 0)] * 3)
cursor.executemany(
    "INSERT INTO Governance.governance_config (config_key, config_value, description) VALUES (?, ?, ?)",
    config_defaults
)
conn.commit()
print("\u2714 governance_config created with defaults:")
display(pd.read_sql("SELECT config_key, config_value FROM Governance.governance_config ORDER BY config_key", conn))

## 9. Logo (optional — needs the Xebia logo PNG uploaded to the attached Lakehouse's Files area)

In [ ]:
from PIL import Image
import io, base64

# Reading the PNG still needs a Lakehouse attached — Warehouses have no Files area.
img = Image.open("/lakehouse/default/Files/Fabric_Monitoring/xebia_logo.png")

if img.mode != 'RGBA':
    img = img.convert('RGBA')

ratio = 150 / img.size[0]
img_small = img.resize((150, int(img.size[1] * ratio)), Image.LANCZOS)

bg = Image.new('RGBA', img_small.size, (255, 255, 255, 255))
bg.paste(img_small, (0, 0), img_small)
img_final = bg.convert('RGB')

buf = io.BytesIO()
img_final.save(buf, format='PNG', optimize=True)
logo_data_uri = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"

# Force plain VARCHAR(MAX) binding for this long string — without this, pyodbc
# auto-detects long parameters as a legacy LOB type, which Fabric Warehouse's UTF-8
# collation rejects with: "Cannot convert to text/ntext ... these legacy LOB types
# do not support UTF-8 or UTF-16 encodings."
cursor.setinputsizes([(pyodbc.SQL_VARCHAR, 0, 0)])
cursor.execute(
    "UPDATE Governance.governance_config SET config_value = ? WHERE config_key = 'logo_url'",
    logo_data_uri
)
conn.commit()
print(f"\u2714 Real Xebia logo saved ({len(logo_data_uri)} chars)")

## 10. Verify all tables

In [ ]:
verify_sql = """
SELECT 'cleanup_tracker' AS table_name, COUNT(*) AS row_count FROM Governance.cleanup_tracker
UNION ALL
SELECT 'cleanup_audit_log', COUNT(*) FROM Governance.cleanup_audit_log
UNION ALL
SELECT 'email_outbox', COUNT(*) FROM Governance.email_outbox
UNION ALL
SELECT 'governance_config', COUNT(*) FROM Governance.governance_config
UNION ALL
SELECT 'workspace_inventory_snapshot', COUNT(*) FROM Governance.workspace_inventory_snapshot
UNION ALL
SELECT 'deleted_items_archive', COUNT(*) FROM Governance.deleted_items_archive
"""
display(pd.read_sql(verify_sql, conn))

## Done

All tables created in `DW_Fabric` under the `Governance` schema. You can now run the
**Governance_Tracker_Update** notebook.

To modify thresholds later:
```sql
UPDATE Governance.governance_config SET config_value = '50' WHERE config_key = 'cleanup_score_threshold'
```

**Note:** don't re-run this notebook to reset data — it drops and recreates every table,
wiping `governance_config` (logo, test-mode setting) and all tracked history. Use targeted
`DELETE`/`UPDATE` statements instead, same as the Lakehouse version.